# argmax-accuracy-eval — ex3: top-k accuracy via logits.topk and any-match

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `argmax-accuracy-eval`. Running the final beacon cell reports progress against the `Eval: argmax accuracy` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Eval: argmax accuracy` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`argmax-accuracy-eval`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "argmax-accuracy-eval"
DD_SUBTOPIC = "Eval: argmax accuracy"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Top-k accuracy — the wider generalization of argmax

Ex1 + ex2 used `logits.argmax(dim=-1)` — top-1 only. Top-k accuracy asks the looser question: 'is the correct label inside the top-`k` highest-scoring predictions?'.

```python
# logits: (B, C). labels: (B,) int64.
topk_preds = logits.topk(k, dim=-1).indices   # (B, k)
correct    = (topk_preds == labels.unsqueeze(-1)).any(dim=-1)   # (B,) bool
acc        = correct.float().mean().item()
```

**Why `topk` instead of `argsort`.** `topk` is O(B*C*log(k)), `argsort` is O(B*C*log(C)). When `k << C` (typical: k=5, C=1000 for ImageNet), topk is asymptotically faster and that's why every classification harness uses it.

**Why `unsqueeze(-1)` + `.any(dim=-1)`.** `labels` is `(B,)`. Broadcasting to `(B, 1)` lets it compare element-wise against `(B, k)`. The `.any` collapses the k-axis: 'did ANY of the top-k slots match?'.

**Top-1 is the k=1 case.** Setting k=1 reduces to ex1's argmax-equality check (after squeezing the size-1 axis). Top-k subsumes top-1.

### Exercise 3 — top-k accuracy via logits.topk and any-match

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `logits.topk(k, dim=-1).indices` followed by a broadcasted equality + `.any(dim=-1)` so the per-example boolean tells you whether the correct label was inside the top-k predictions.
> Keywords: accuracy, top-k, topk, classification
> ```

**KCs targeted:** `topk-indices-along-class-dim`, `any-match-collapses-k-axis`

Implement `ex3_topk_accuracy(logits, labels, k)`. The top-k classification accuracy metric.

Inputs:
- `logits`: `(B, C)` float tensor.
- `labels`: `(B,)` int64 tensor with values in `[0, C)`.
- `k`: int, `1 <= k <= C`.

Steps:

1. Get the indices of the top-k highest-scoring classes per row: `topk_idx = logits.topk(k, dim=-1).indices` — shape `(B, k)`.
2. Compare against labels: broadcast `labels` to `(B, 1)` via `labels.unsqueeze(-1)` and compare element-wise against `topk_idx` — gives a `(B, k)` bool tensor.
3. Collapse the k-axis with `.any(dim=-1)` — `(B,)` bool: True iff any of the top-k slots matched.
4. Convert to float and average: `.float().mean().item()`.

Return a Python float in `[0.0, 1.0]`.

In [ ]:
def ex3_topk_accuracy(logits, labels, k: int) -> float:
    """Fraction of examples where the correct label is in the top-k predictions."""
    raise NotImplementedError()


def _test_ex3():
    # === Hand-crafted: 4 examples, 5 classes ===
    # logits row 0: argmax=2 (largest), top-3 = {2, 1, 4}
    # logits row 1: argmax=0, top-3 = {0, 3, 1}
    # logits row 2: argmax=4, top-3 = {4, 2, 0} (label=0 in top-3 but NOT in top-2)
    # logits row 3: argmax=1, top-3 = {1, 4, 0}
    logits = t.tensor([
        [0.1, 0.5, 2.0, 0.0, 0.4],
        [3.0, 0.5, 0.1, 1.0, 0.0],
        [0.3, 0.0, 0.8, 0.2, 2.5],
        [0.4, 2.2, 0.1, 0.0, 1.5],
    ])
    labels = t.tensor([2, 3, 0, 4])  # correct slots

    # === Top-1 == argmax check ===
    # argmax row-by-row: 2, 0, 4, 1.  Match labels (2,3,0,4)? 1, 0, 0, 0 → 0.25.
    assert abs(ex3_topk_accuracy(logits, labels, k=1) - 0.25) < 1e-6, (
        f'k=1 should match argmax accuracy: expected 0.25, got {ex3_topk_accuracy(logits, labels, 1)}'
    )

    # === Top-3 ===
    # row 0: label=2 in {2,1,4} → True
    # row 1: label=3 in {0,3,1} → True
    # row 2: label=0 in {4,2,0} → True
    # row 3: label=4 in {1,4,0} → True
    # → all four match → 1.0
    assert ex3_topk_accuracy(logits, labels, k=3) == 1.0, (
        f'k=3 should give 1.0; got {ex3_topk_accuracy(logits, labels, 3)}'
    )

    # === Top-2 ===
    # row 0: label=2 in {2,1} → True
    # row 1: label=3 in {0,3} → True
    # row 2: label=0 in {4,2} → False
    # row 3: label=4 in {1,4} → True
    # → 3/4 = 0.75
    assert abs(ex3_topk_accuracy(logits, labels, k=2) - 0.75) < 1e-6, (
        f'k=2 expected 0.75; got {ex3_topk_accuracy(logits, labels, 2)}'
    )

    # === Top-5 (=C) always 1.0 ===
    assert ex3_topk_accuracy(logits, labels, k=5) == 1.0, (
        f'k=C should give 1.0; got {ex3_topk_accuracy(logits, labels, 5)}'
    )

    # === Return type is Python float ===
    out = ex3_topk_accuracy(logits, labels, k=1)
    assert isinstance(out, float), f'must return Python float, got {type(out)}'
    assert 0.0 <= out <= 1.0

    # === Monotone in k ===
    # top-k accuracy is non-decreasing in k for any fixed (logits, labels).
    t.manual_seed(1)
    B, C = 32, 10
    L = t.randn(B, C)
    y = t.randint(0, C, (B,))
    accs = [ex3_topk_accuracy(L, y, k=k) for k in [1, 2, 3, 5, 7, 10]]
    for i in range(len(accs) - 1):
        assert accs[i] <= accs[i+1] + 1e-9, (
            f'top-k accuracy must be non-decreasing in k: {accs}'
        )
    assert accs[-1] == 1.0, f'k=C should give 1.0; got accs={accs}'

    # === Perfect predictions → 1.0 at every k ===
    # Construct logits where argmax row-i = labels[i] with huge margin.
    y = t.tensor([3, 1, 4, 0])
    L = t.zeros(4, 5)
    L[t.arange(4), y] = 10.0
    for k_test in [1, 2, 3, 4, 5]:
        assert ex3_topk_accuracy(L, y, k_test) == 1.0, f'perfect at k={k_test} should be 1.0'
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_topk_accuracy(logits, labels, k):
    topk_idx = logits.topk(k, dim=-1).indices       # (B, k)
    correct = (topk_idx == labels.unsqueeze(-1)).any(dim=-1)  # (B,) bool
    return correct.float().mean().item()
```

**Why `topk` over `argsort`.** `argsort` is O(C log C); `topk` is O(C log k). For ImageNet-style `C=1000, k=5`, `topk` is ~30x fewer comparisons per row and is what every benchmark harness uses.

**Why `unsqueeze(-1)` matters.** Without it, `labels` is `(B,)` and `topk_idx` is `(B, k)` — broadcasting would attempt `(B,) vs (B, k)` which raises (broadcasting from the right, `B != k`). Unsqueezing gives `(B, 1) vs (B, k)` → valid broadcast.

**Top-1 == argmax sanity.** Setting k=1 reduces to ex1's metric after squeezing the size-1 axis — a useful self-consistency check and the first test we run.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()